In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
from urllib.parse import urljoin
from urllib.parse import urlparse
import os
from pathlib import Path
import pdfplumber


In [2]:
## %pip install --upgrade certifi requests urllib3

In [3]:
url = "https://www.mumbwacouncil.gov.zm/"

In [4]:
response = requests.get(url,timeout=30 ,verify=False)

C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [5]:
print(response.status_code)
print(response.url)
print(response.text[:2000])

200
https://www.mumbwacouncil.gov.zm/
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1.0, viewport-fit=cover" />		<title>Mumbwa Town Council &#8211; Mumbwa</title>
<meta name='robots' content='max-image-preview:large' />
<link rel='dns-prefetch' href='//s.w.org' />
<link rel="alternate" type="application/rss+xml" title="Mumbwa Town Council &raquo; Feed" href="https://www.mumbwacouncil.gov.zm/?feed=rss2" />
<link rel="alternate" type="application/rss+xml" title="Mumbwa Town Council &raquo; Comments Feed" href="https://www.mumbwacouncil.gov.zm/?feed=comments-rss2" />
<script>
window._wpemojiSettings = {"baseUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/72x72\/","ext":".png","svgUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/svg\/","svgExt":".svg","source":{"concatemoji":"https:\/\/www.mumbwacouncil.gov.zm\/wp-includes\/js\/wp-emoji-release.min.js?ver=5.9"}};
/*! This file is auto-generate

In [6]:
print("<table" in response.text.lower())

False


In [7]:
soup = BeautifulSoup(response.text, "html.parser")

print("Tables:", len(soup.find_all("table")))
print("Scripts:", len(soup.find_all("script")))
print("Links:", len(soup.find_all("a")))

Tables: 0
Scripts: 50
Links: 80


In [8]:
# %pip install --upgrade certifi requests urllib3


In [9]:

soup = BeautifulSoup(response.text, "html.parser")

print("Tables found:", len(soup.find_all("table")))
print("Links found:", len(soup.find_all("a")))
print("Scripts found:", len(soup.find_all("script")))

Tables found: 0
Links found: 80
Scripts found: 50


In [10]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(url, link["href"])

    links.append({
        "text": text,
        "url": href
    })

links_df = pd.DataFrame(links)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(links_df)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,https://www.mumbwacouncil.gov.zm/#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [11]:
tracker_url = "https://www.mumbwacouncil.gov.zm/?page_id=932"

tracker_response = requests.get(
    tracker_url,
    verify=False,
    timeout=30
)

tracker_soup = BeautifulSoup(tracker_response.text, "html.parser")

print("Status:", tracker_response.status_code)
print("Tables:", len(tracker_soup.find_all("table")))
print("Links:", len(tracker_soup.find_all("a")))

C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Tables: 0
Links: 114


In [12]:
tracker_links = []

for link in tracker_soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(tracker_url, link["href"])

    tracker_links.append({
        "text": text,
        "url": href
    })

tracker_links_df = pd.DataFrame(tracker_links)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(tracker_links_df)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,https://www.mumbwacouncil.gov.zm/?page_id=932#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [13]:
cdf_pdfs = tracker_links_df[
    tracker_links_df["url"].str.contains(
        r"\.pdf",
        case=False,
        na=False
    )
].copy()

display(cdf_pdfs)

,text,url
66,2025 Approved Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Mumbwa-Central.pdf
67,2025 Proposed Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf
69,2025 Approved CDF Skills Development Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf
71,2025 Approved CDF Secondary Boarding School Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf
73,2025 Approved CDF Empowerment Grants for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf
94,2025 Approved Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Nangoma-Constituency.pdf
95,2025 Proposed Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf
97,2025 Approved CDF Empowerment Grants for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf
100,2025 Approved CDF Skills Development Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf
102,2025 Approved CDF Secondary Boarding School Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf


In [14]:
mumbwa_cdf_pdfs = raw_pdf_dir
download_folder = raw_pdf_dir
os.makedirs(download_folder, exist_ok=True)

for _, row in cdf_pdfs.iterrows():

    url = row["url"]

    filename = os.path.basename(
        urlparse(url).path
    )

    filepath = os.path.join(
        download_folder,
        filename
    )

    print(f"Downloading: {filename}")

    pdf_response = requests.get(
        url,
        verify=False,
        timeout=100
    )

    print("Status:", pdf_response.status_code)

    if pdf_response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(pdf_response.content)

        print("Saved:", filepath)
    else:
        print("FAILED:", url)

    print("-" * 80)


Downloading: 2025-Approved-Community-Projects-Mumbwa-Central.pdf


C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Saved: C:\Users\Moses Chaswala\Desktop\mumbwa_cdf_pdfs\Mumbwa-Dataset\mumbwa-town-council-dataset\data\raw\pdfs\2025-Approved-Community-Projects-Mumbwa-Central.pdf
--------------------------------------------------------------------------------
Downloading: NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf


C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Saved: C:\Users\Moses Chaswala\Desktop\mumbwa_cdf_pdfs\Mumbwa-Dataset\mumbwa-town-council-dataset\data\raw\pdfs\NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf
--------------------------------------------------------------------------------
Downloading: 2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf


...

Status: 200
Saved: C:\Users\Moses Chaswala\Desktop\mumbwa_cdf_pdfs\Mumbwa-Dataset\mumbwa-town-council-dataset\data\raw\pdfs\2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf
--------------------------------------------------------------------------------
Downloading: 2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf


C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Saved: C:\Users\Moses Chaswala\Desktop\mumbwa_cdf_pdfs\Mumbwa-Dataset\mumbwa-town-council-dataset\data\raw\pdfs\2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf
--------------------------------------------------------------------------------
Downloading: 2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf


C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Saved: C:\Users\Moses Chaswala\Desktop\mumbwa_cdf_pdfs\Mumbwa-Dataset\mumbwa-town-council-dataset\data\raw\pdfs\2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf
--------------------------------------------------------------------------------


In [15]:
pdf_files = [
    str(pdf_file)
    for pdf_file in sorted(mumbwa_cdf_pdfs.glob("*.pdf"))
]

print("PDFs found:", len(pdf_files))

for pdf in pdf_files:
    print(os.path.basename(pdf))


PDFs found: 10
2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf
2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf
2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf
2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf
2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf
2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf
2025-Approved-Community-Projects-Mumbwa-Central.pdf
2025-Approved-Community-Projects-Nangoma-Constituency.pdf
NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf
NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf


In [16]:
for pdf_file in pdf_files:

    print("\n" + "#" * 100)
    print(os.path.basename(pdf_file))
    print("#" * 100)

    with pdfplumber.open(pdf_file) as pdf:

        total_tables = 0

        for page_number, page in enumerate(pdf.pages, start=1):

            tables = page.extract_tables()

            if tables:
                print(
                    f"Page {page_number}: "
                    f"{len(tables)} table(s)"
                )

                total_tables += len(tables)

        print("TOTAL TABLES:", total_tables)


####################################################################################################
2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf
####################################################################################################
TOTAL TABLES: 0

####################################################################################################
2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf
####################################################################################################
TOTAL TABLES: 0

####################################################################################################
2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf
####################################################################################################
TOTAL TABLES: 0

####################################################################################################
2025-Approved-CDF-Secondary-Boarding-School-Bursaries

In [5]:
pdf1_path = "../data/intermediate/reconstructed_pdfs/2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf"


In [7]:
for i, df_page in enumerate(all_tables, start=1):
    print(f"Table {i}:")
    print(df_page.columns.tolist())
    print("Duplicate columns:", df_page.columns[df_page.columns.duplicated()].tolist())
    print()

Table 1:
['2', 'Construction of 1x2 Classroom Block at\nKamilambo Primary School', 'Education', 'Construction', 'Mumbwa', 'Mumbwa', 'Kamilambo', 'Kamilambo', '2025', '1x2 Classroom Block', 'Approved']
Duplicate columns: ['Mumbwa', 'Kamilambo']

Table 2:
['No.', 'Project Name', 'Sector', 'Type of Project', 'District', 'Constituency', 'Ward', 'Project Site/Location', 'Year Funded', 'Work package (inclusive of CDF Branding)', 'Comments/Remarks']
Duplicate columns: []



In [8]:
columns = [
    "No.",
    "Project Name",
    "Sector",
    "Type of Project",
    "District",
    "Constituency",
    "Ward",
    "Project Site/Location",
    "Year Funded",
    "Work package (inclusive of CDF Branding)",
    "Comments/Remarks"
]

In [27]:
all_tables = []

columns = [
    "No.",
    "Project Name",
    "Sector",
    "Type of Project",
    "District",
    "Constituency",
    "Ward",
    "Project Site/Location",
    "Year Funded",
    "Work package (inclusive of CDF Branding)",
    "Comments/Remarks"
]

with pdfplumber.open(pdf1_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):

        tables = page.extract_tables()

        for table in tables:
            if not table:
                continue

            # Convert extracted table to DataFrame without assuming first row is header
            df_page = pd.DataFrame(table)

            # Remove header row if this page contains the actual header
            if df_page.iloc[0].tolist() == columns:
                df_page = df_page.iloc[1:]

            # Apply the correct column names
            if df_page.shape[1] == len(columns):
                df_page.columns = columns

                # Optional: record source page
                df_page["source_page"] = page_num

                all_tables.append(df_page)
            else:
                print(
                    f"Skipped table on page {page_num}: "
                    f"expected {len(columns)} columns, "
                    f"found {df_page.shape[1]}"
                )

df1 = pd.concat(all_tables, ignore_index=True)

df1

,No.,Project Name,Sector,Type of Project,District,Constituency,Ward,Project Site/Location,Year Funded,Work package (inclusive of CDF Branding),Comments/Remarks,source_page
0,1,Construction of 1x2 Classroom Block at\nMalomb...,Education,Construction,Mumbwa,Mumbwa,Shimbizhi,Malombe Primary School,2025,1x2 Classroom Block,Approved,1
1,2,Construction of 1x2 Classroom Block at\nKamila...,Education,Construction,Mumbwa,Mumbwa,Kamilambo,Kamilambo,2025,1x2 Classroom Block,Approved,1
2,3,Completion of Kabawa Rural Health Post,Health,Completion,Mumbwa,Mumbwa,Chibolyo,Kabawa Rural Health Post,2025,1No. Health Post\nIncinerator,Approved,1
3,4,Completion of a Maternity Wing & Water\nReticu...,Health,Construction &\nReticulation,Mumbwa,Mumbwa,Makebo,Kabwanga Rural Health\nPost,2025,Completion of Maternity Wing\nWater reticulati...,Approved,1
4,5,Construction of 1x2 Semi detached\nTeachers St...,Education,Construction,Mumbwa,Mumbwa,Kalwanyembe,Kalenda Secondary School,2025\n2025,1No. Semi detached Staff house\nWater Reticula...,Approved,1
5,6,Installation of a roofing system for 1x3\nClas...,Education,Construction,Mumbwa,Mumbwa,Kalwanyembe,Kitumba Community School,2025,Installation of a roofing system for 1x3\nClas...,Approved,1
6,7,Construction of Vet offices/Laboratory,"Fisheries, Livestock and\nVeterinary serves",Construction,Mumbwa,Mumbwa,Mupona,Veterinery offices/Mumbwa,2025,1No. Office block with 12No. offices including...,Approved,1
7,8,Rehabilitation of Nalusanga Market,Infrastructure,Construction,Mumbwa,Mumbwa,Nalusanga,Nalusanga Market,2025,Rehabilitation of 15mx12m shelter with+48\nsel...,Approved,1
8,3,Kawena,Roads Infrastructure,Rehabilitation,12.8km,Mumbwa,Nalusanga,Nalusanga,2025,"Grading & Gravelling,30m",Approved,2
9,4,Chibuluma,Roads Infrastructure,Rehabilitation,41km,Mumbwa,Chibolyo,Chibolyo,2025,"Grading & Gravellling,line and mitre drains",Approved,2


In [28]:
df1.to_csv("../data/intermediate/reconstructed_pdfs/2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf")


In [40]:
pdf2_path = "../data/intermediate/reconstructed_pdfs/2025_Approved_Community_Projects_Nangoma_Constituency_Recreated_Tables.pdf"

tables2 = read_pdf_tables(pdf2_path)

In [41]:
print("Number of tables:", len(tables2))

for i, table in enumerate(tables2, start=1):
    print(f"\nTable {i}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

Number of tables: 3

Table 1
Shape: (12, 9)
Columns: ['No', 'Project Name', 'Project Description', 'Constituency', 'Ward', 'Project Site/Location', 'Scope of Works', 'source_page', 'source_table']

Table 2
Shape: (1, 9)
Columns: ['No', 'Project Name', 'Project Description', 'Constituency', 'Ward', 'Project Site/Location', 'Scope of Works', 'source_page', 'source_table']

Table 3
Shape: (3, 8)
Columns: ['SN', 'Name of Roads', 'Constituency', 'Ward', 'Distance (Km)', 'Scope of Work', 'source_page', 'source_table']


In [42]:
for i, table in enumerate(tables2):
    print(f"TABLE {i}")
    display(table.head())

TABLE 0


,No,Project Name,Project Description,Constituency,Ward,Project Site/Location,Scope of Works,source_page,source_table
0,1,Construction of Kantengwa Bridge\n(Additional),Construction of kantengwa Bridge\n(Additional),Nangoma,Matala,Kantengwa,Additional Funding,1,1
1,2,Grading of 31km Nachibila to Nalubanda,Grading of 31km Nachibila to Nalubanda,Nangoma,Myooye Shichanzu\nNalubanda,Myooye Shichanzu\nNalubanda,Additional Funding,1,1
2,3,Procurement of Roller Compactor,270 litre Diesel Full hydraulic steering gear ...,Nangoma,All,All,270 litre Diesel Full hydraulic steering gear ...,1,1
3,4,Procurement of Tipper Truck,Procurement of Tipper Truck,Nangoma,All,All,Procurement of Tipper Truck,1,1
4,5,Upgrading of Nakabu secondary School by\nconst...,Construction of a 1*3CRB with procurement of 9...,Nangoma,Nangoma,Nakabu Secondary\nSchool,Construction of a 1*3CRB with procurement of 9...,1,1


TABLE 1


,No,Project Name,Project Description,Constituency,Ward,Project Site/Location,Scope of Works,source_page,source_table
0,13,Rehabilitation of Myooye Secondary School staf...,Replacement of roofing truss system,Nangoma,Myooye,Myooye Secondary\nschool,Rehabilitation of Myooye Secondary School staf...,2,1


TABLE 2


,SN,Name of Roads,Constituency,Ward,Distance (Km),Scope of Work,source_page,source_table
0,1,Nakabu Road,Nangoma,Nangoma,14,"Grading and Gravelling,line and mitre\ndrain",2,2
1,2,Luili Road,Nangoma,Mumba,25,"Grading, gravelling and installtion of\nCulverts",2,2
2,3,Muchabi Road,Nangoma,Nalubanda,12.5,"Grading, gravelling and installtion of\nCulverts",2,2


In [43]:
datasets2 = combine_matching_tables(tables2)

print("Different datasets found:", len(datasets2))

for i, df in enumerate(datasets2, start=1):
    print(f"Dataset {i}: {df.shape}")
    print(df.columns.tolist())

NameError: name 'combine_matching_tables' is not defined